# Milestone 1: Exploratory Data Analysis
## Predicting Potentially Hazardous Asteroids (NASA JPL Asteroid Dataset)

Team: Team 1 | Members: Samuel T, Ishita C, Sebastian A | Course: Machine Learning, Fall 2026

**How to use this notebook**
1. Put the Kaggle file `dataset.csv` in the same folder as this notebook (or change `DATA_PATH` in the next cell).
2. Restart the kernel and choose Run All. Every code cell must show its output before you save and export.
3. All plots are saved as PNG files in a `figures` folder. Paste the key ones into Section 3.4 of the report.
4. Cells that say "Your notes" are for you to write 2 to 3 sentences in your own words. Those sentences are the insights the rubric grades.

In [ ]:
# Standard libraries and plotting
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split

# Show all columns and use a wide display so tables are not cut off
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

DATA_PATH = "dataset.csv"   # change this if your file has a different name or folder
FIG_DIR = "figures"         # every plot is saved here as a PNG for the report
RANDOM_STATE = 42           # fixed seed so samples and splits are reproducible
os.makedirs(FIG_DIR, exist_ok=True)


def save_fig(name):
    """Tidy the layout, save the current figure as a PNG, and show it."""
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches="tight")
    plt.show()

## 1. Load the data

In [ ]:
# low_memory=False stops pandas from guessing column types in chunks, which avoids mixed-type warnings
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Shape (rows, columns):", df.shape)
display(df.head())

## 2. Column types and data dictionary check

Report location: Section 3.1 (size, numeric versus text columns) and Section 3.3 (key variables table).

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
text_cols = [c for c in df.columns if c not in numeric_cols]
print(len(numeric_cols), "numeric columns and", len(text_cols), "text columns")
print("Text columns:", text_cols)

# One row per column: type, how many values exist, how many are missing, how many distinct values
dictionary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
    "example_value": df.iloc[0].astype(str),
})
display(dictionary)

## 3. Target variable: `pha`

Report location: Section 3.1 (class balance) and Section 3.4 (target distribution row).

In [ ]:
n_n = int((df["pha"] == "N").sum())
n_y = int((df["pha"] == "Y").sum())
n_missing = int(df["pha"].isna().sum())

target_table = pd.DataFrame(
    {"count": [n_n, n_y, n_missing]},
    index=["N (not hazardous)", "Y (hazardous)", "missing"],
)
target_table["percent"] = (target_table["count"] / len(df) * 100).round(3)
display(target_table)

# Only rows with a label can be used for supervised learning
labeled = df[df["pha"].notna()].copy()
labeled["y"] = (labeled["pha"] == "Y").astype(int)   # 1 = hazardous, 0 = not hazardous
print("Labeled rows:", len(labeled))
print("Hazardous share of labeled rows (%):", round(labeled["y"].mean() * 100, 3))
print("About 1 hazardous object per", round(len(labeled) / n_y), "labeled objects")
print("Accuracy of a model that always predicts N (%):", round((1 - labeled["y"].mean()) * 100, 2))

# Bar chart on a log scale so the small Y bar is still visible
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["N", "Y", "missing"], [n_n, n_y, n_missing])
ax.set_yscale("log")
ax.set_ylabel("Number of objects (log scale)")
ax.set_title("Target variable: pha")
save_fig("target_distribution.png")

**Your notes:** Why does the always-predict-N accuracy above make accuracy a poor metric for this project?

## 4. Missing values

Report location: Section 3.4 (missing values row) and Section 3.5 (data quality).

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct[missing_pct > 0].round(2).to_frame("missing_percent"))

# Bar chart of the 20 columns with the most missing values
top_missing = missing_pct[missing_pct > 0].head(20)
fig, ax = plt.subplots(figsize=(8, 5))
top_missing[::-1].plot(kind="barh", ax=ax)
ax.set_xlabel("Missing (%)")
ax.set_title("Columns with the most missing values")
save_fig("missing_values.png")

# Do the unlabeled rows overlap with the rows that lack moid and the sigma (uncertainty) columns?
sigma_cols = [c for c in df.columns if c.startswith("sigma_")]
unlabeled = df["pha"].isna()
print("pha missing:", int(unlabeled.sum()))
print("moid missing:", int(df["moid"].isna().sum()))
print("Rows missing BOTH pha and moid:", int((unlabeled & df["moid"].isna()).sum()))
print("Unlabeled rows that have no value in any sigma column:", int(df.loc[unlabeled, sigma_cols].isna().all(axis=1).sum()))

## 5. Summary statistics and distributions

Report location: Section 3.4 (summary statistics and distributions rows).

In [ ]:
display(df[numeric_cols].describe().T)

# Skewness: values far from 0 mean a long tail on one side (a candidate for a log transform)
skew = df[numeric_cols].skew().sort_values(key=abs, ascending=False)
print("Most skewed numeric columns:")
display(skew.head(10).round(2).to_frame("skewness"))

In [ ]:
# Histograms of key variables. Extreme values would squash the plot, so each panel
# shows only the 1st to 99th percentile (the table above still reports the full range).
dist_cols = ["e", "i", "q", "a", "H", "per_y"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), dist_cols):
    s = df[col].dropna()
    lo, hi = s.quantile([0.01, 0.99])
    ax.hist(s[(s >= lo) & (s <= hi)], bins=60)
    ax.set_title(f"{col} (1st to 99th percentile)")
    ax.set_ylabel("count")
save_fig("distributions.png")

In [ ]:
# Orbit classes (log scale because the main belt dominates)
class_counts = df["class"].value_counts()
display(class_counts.to_frame("count"))

fig, ax = plt.subplots(figsize=(8, 4))
class_counts.plot(kind="bar", ax=ax)
ax.set_yscale("log")
ax.set_ylabel("Number of objects (log scale)")
ax.set_title("Orbit class counts")
save_fig("orbit_classes.png")

**Your notes:** Describe the shape of `a`, `e`, and `i` in the histograms. Which orbit class holds most objects, and what does that mean for the rare hazardous class?

## 6. Features by hazard class

Report location: Section 3.4 (features by class row).

In [ ]:
box_cols = ["a", "e", "i", "q"]
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, col in zip(axes, box_cols):
    groups = [labeled.loc[labeled["pha"] == g, col].dropna() for g in ["N", "Y"]]
    ax.boxplot(groups, showfliers=False)   # outliers hidden so the boxes stay readable
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["N", "Y"])
    ax.set_title(f"{col} by pha")
save_fig("features_by_class.png")

print("Median by class (medians resist the extreme values):")
display(labeled.groupby("pha")[box_cols].median().round(3))
print("Mean by class:")
display(labeled.groupby("pha")[box_cols].mean().round(3))

# Which orbit classes contain hazardous objects?
hazard_by_class = pd.crosstab(labeled["class"], labeled["pha"])
hazard_by_class["percent_Y"] = (hazard_by_class["Y"] / (hazard_by_class["N"] + hazard_by_class["Y"]) * 100).round(3)
display(hazard_by_class)

**Your notes:** How do hazardous objects differ from the rest in perihelion distance `q`, eccentricity `e`, and inclination `i`? Does that match orbital reasoning (objects that come close to Earth's orbit)?

## 7. How the label is defined (leakage check)

NASA defines a hazardous asteroid as one with moid <= 0.05 AU and H <= 22. This cell tests how well that rule explains `pha` in this file.

Report location: Section 3.4 (label dependence row) and Section 2.1.

In [ ]:
moid_ok = (labeled["moid"] <= 0.05).rename("moid<=0.05")
h_ok = (labeled["H"] <= 22).rename("H<=22")   # a missing H counts as False here
display(pd.crosstab(labeled["pha"], [moid_ok, h_ok]))

# Rows where the label disagrees with the rule
rule = moid_ok & h_ok
exceptions = labeled[(labeled["pha"] == "Y") != rule]
print("Exceptions (label disagrees with the rule):", len(exceptions))
print("Share of labeled rows (%):", round(len(exceptions) / len(labeled) * 100, 4))
print("Exceptions where H is missing:", int(exceptions["H"].isna().sum()))
display(exceptions[["full_name", "pha", "H", "moid", "diameter", "class"]].sort_values(["pha", "H"]))

## 8. Data quality checks

Report location: Section 3.5.

In [ ]:
print("Duplicate id values:", int(df["id"].duplicated().sum()))
print("Duplicate full_name values:", int(df["full_name"].duplicated().sum()))
print("Fully duplicated rows:", int(df.duplicated().sum()))

# Values that need a decision (hyperbolic orbits are valid objects, not errors)
print("\nOrbit checks:")
print("e >= 1 (hyperbolic, unbound orbit):", int((df["e"] >= 1).sum()))
print("a < 0 (same cause as e > 1):", int((df["a"] < 0).sum()))
print("ma outside 0 to 360 degrees:", int(((df["ma"] < 0) | (df["ma"] > 360)).sum()))
print("i outside 0 to 180 degrees:", int(((df["i"] < 0) | (df["i"] > 180)).sum()))
print("rms greater than 10 (very poor orbit fit):", int((df["rms"] > 10).sum()))

# The sigma (uncertainty) columns: how often are they exactly zero?
print("\nShare of rows where each sigma column equals 0:")
display((df[sigma_cols] == 0).mean().round(3).to_frame("share_equal_zero"))

# Near-constant columns carry almost no information for a model
print("Near-constant columns:")
for col in ["epoch", "epoch_mjd", "epoch_cal", "equinox"]:
    top_share = df[col].value_counts(normalize=True).iloc[0]
    print(f"  {col}: {df[col].nunique()} distinct values, the most common covers {top_share * 100:.1f}% of rows")

## 9. Correlation analysis

Report location: Section 3.4 (correlation row) and Section 3.5 (redundant columns).

Pearson correlation is easily dominated by a few extreme values, so we also compute Spearman (rank) correlation on a random sample of 100,000 rows.

In [ ]:
def heatmap(c, title, name):
    """Draw a labeled correlation heatmap and save it."""
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(c.values, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(c)))
    ax.set_xticklabels(c.columns, rotation=90)
    ax.set_yticks(range(len(c)))
    ax.set_yticklabels(c.index)
    for r in range(len(c)):
        for k in range(len(c)):
            ax.text(k, r, f"{c.iloc[r, k]:.2f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title)
    save_fig(name)


def top_pairs(c, threshold=0.9):
    """List each pair of columns once (upper triangle) with absolute correlation above the threshold."""
    upper = c.abs().where(np.triu(np.ones(c.shape, dtype=bool), k=1))
    pairs = upper.stack()
    return pairs[pairs > threshold].sort_values(ascending=False).round(3)


corr_cols = ["H", "diameter", "albedo", "e", "a", "q", "i", "om", "w", "ma", "ad", "n", "per_y", "moid", "rms"]

pearson = df[corr_cols].corr()
spearman = df[corr_cols].sample(min(100_000, len(df)), random_state=RANDOM_STATE).corr(method="spearman")

heatmap(pearson, "Pearson correlation", "corr_pearson.png")
heatmap(spearman, "Spearman correlation (100,000-row sample)", "corr_spearman.png")

print("Column pairs with |Pearson| > 0.9 (all numeric columns):")
display(top_pairs(df[numeric_cols].corr()).to_frame("abs_correlation"))

# Correlation of each variable with the target (1 = hazardous). H and moid are expected to rank high.
target_corr = pd.DataFrame({
    "pearson": labeled[corr_cols].corrwith(labeled["y"]),
    "spearman": labeled[corr_cols].corrwith(labeled["y"], method="spearman"),
}).round(3)
print("Correlation with the target:")
display(target_corr.sort_values("spearman", key=abs, ascending=False))

In [ ]:
# Perihelion distance q versus moid: is q nearly a copy of moid?
pair = df[["q", "moid"]].dropna()
print("Pearson correlation of q and moid:", round(pair["q"].corr(pair["moid"]), 5))
print("Median of (q - moid) in AU:", round((pair["q"] - pair["moid"]).median(), 3))

sample = pair.sample(min(100_000, len(pair)), random_state=RANDOM_STATE)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(sample["q"], sample["moid"], s=2, alpha=0.2)
ax.set_xlim(0, 6)
ax.set_ylim(0, 6)
ax.set_xlabel("q: perihelion distance (AU)")
ax.set_ylabel("moid (AU)")
ax.set_title("q versus moid (100,000-row sample)")
save_fig("q_vs_moid.png")

**Your notes:** Which columns are redundant copies of each other, and which one will you keep from each group? What does the q versus moid plot imply for a model that is not given moid directly?

## 10. Preparation preview and stratified split

This cell applies the plan from Section 4 of the report: remove identifiers, near-constant columns, label-defining columns, and heavily missing columns, drop rows with no label, and split with stratification.

Report location: Sections 4.1 and 4.4.

In [ ]:
id_cols = ["id", "spkid", "full_name", "pdes", "name", "prefix", "orbit_id"]
constant_cols = ["equinox", "epoch", "epoch_mjd", "epoch_cal", "tp", "tp_cal"]
leakage_cols = ["moid", "moid_ld", "H", "diameter"]          # define or are computed from the label rule
sparse_cols = ["albedo", "diameter_sigma"]                   # more than 85% missing
drop_for_primary = id_cols + constant_cols + leakage_cols + sparse_cols + sigma_cols

X = labeled.drop(columns=drop_for_primary + ["pha", "y"])
y = labeled["y"]
print("Feature columns for the primary model:", X.columns.tolist())
print("Rows used:", len(X))

print("\nMissing values left in the features (%):")
left_missing = (X.isna().mean() * 100).round(3)
display(left_missing[left_missing > 0].to_frame("missing_pct"))

# 70 percent train, 15 percent validation, 15 percent test, stratified on the target
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_val), len(y_test)],
    "hazardous": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
}, index=["train", "validation", "test"])
split_summary["hazardous_percent"] = (split_summary["hazardous"] / split_summary["rows"] * 100).round(3)
display(split_summary)

## 11. Summary of findings (write in your own words)

Fill these in before you export, then copy the points into the report:
- **Target and balance:**
- **Missing data and what we will do about it:**
- **Label rule and leakage (moid, H, diameter, and the q relationship):**
- **Redundant or near-constant columns:**
- **Outliers and values that need a decision:**
- **What surprised us:**